# Task 12 Solutions — Scope, Iterators, Generators & Decorators

This notebook contains the complete worked solutions to every problem on the Task 12 sheet. Where the task notebook asked you to write code from scratch, here each question is followed by a full runnable program and the exact output it produces. Your job now is not to guess the solution but to **read it, run it, and make sure you could have written it** — so use the short explanations that precede each program as a guided review of the two big ideas behind the sheet: iterator machinery, and decorator machinery.

The first theme is **namespaces and scope**. A namespace is simply a dictionary of names — a class body has its own namespace holding attributes and methods, and every instance also has one holding its instance variables. When you ask Python for `instance.attribute`, it looks in the instance namespace first, then in the class hierarchy. The exercises Q1–Q3 make you *print* the namespaces of a class and of an object with `ClassName.__dict__` and `instance.__dict__`, which is one of the clearest ways to *see* scoping in action. Q3 also revisits recursion: a counter is threaded through a recursive `gcd` so the number of function calls is reported.

The second theme is **iterators and generators**. An iterator is any object that implements `__iter__` (returning itself) and `__next__` (returning the next value or raising `StopIteration`); Q4 and Q5 rebuild these protocols by hand for a homemade `enumerate` and a "circle" that cycles a sequence. Q6 switches to a **generator function** — a function containing `yield` — which is the lazy, stateful cousin of a fully written iterator class. The third theme is **decorators**: functions that take a function and return a new, enhanced function. Q7–Q10 chain simple wrappers, print return values, call a function twice, and double a return value, always tested with an `assert`.

Run every cell top to bottom so the outputs line up with what you see below. By the last cell you should feel confident explaining *why* each solution works — not just that it does.

### `Q1:` Write `Person` Class as given below and then display it's namespace.



```
Class Name - Person

Attributes:
name - public
state - public
city - private
age - private

Methods:
address - public
It give address of the person as "<name>, <city>, <state>"
```

### Q1 — a private-attribute `Person` class, then its namespace

The solution defines a class `Person` whose `__init__` stores `name` and `state` normally, while `city` and `age` are kept private with a leading double underscore (`self.__city`). The double underscore triggers **name mangling**: inside the class body the attribute is renamed to `_Person__city`, which is what you later see in the printed namespace. Public getters and setters (`set_city`, `get_city`, `set_age`, `get_age`) are the sanctioned way to touch the private data, and `address()` formats the person's details with an f-string.

The interesting part is the loop:

```python
for i in Person.__dict__:
    print(i)
```

`Person.__dict__` is the **class namespace** — a dictionary mapping every attribute name defined in the class to its value. Because the class attribute `__name__` was added automatically *inside the class body* at definition time, `__module__`, `__init__`, `set_city`, `get_city`, `set_age`, `get_age`, `address`, `__dict__`, `__weakref__`, and `__doc__` all appear. Notice `_Person__city` and `_Person__age` do *not* appear here — they are not created until an instance is built and `self.__city` is assigned, so they live in the *instance* namespace instead.

### Why this works
- `Class.__dict__` reveals everything about where a class keeps its machinery.
- Name mangling (`__x` → `_Class__x`) is why private attributes show up with the mangled name later.

In [ ]:
#Write your code here
class Person:
    def __init__(self, name, state):
        self.name = name
        self.state = state
        self.__city = None
        self.__age = None
    def set_city(self, city):
        self.__city = city
    def get_city(self):
        return self.__city
    def set_age(self, age):
        self.__age = age
    def get_age(self):
        return self.__age
    def address(self):
        return f'{self.name}, {self.__city}, {self.state}'

for i in Person.__dict__:
    print(i)

__module__
__init__
set_city
get_city
set_age
get_age
address
__dict__
__weakref__
__doc__


### `Q2:` Write a program to show namespace of object/instance of above(Person) class.

### Q2 — the instance namespace, where your data actually lives

This program creates one `Person` instance with `name` and `state`, then programs the private values through setters, prints the formatted address, and finally walks through the **instance namespace**:

```python
for i in p.__dict__:
    print(i)
```

`p.__dict__` is a per-object namespace holding only the instance variables — `name`, `state`, `_Person__city`, `_Person__age`. Compare this with Q1: the *class* namespace listed methods and no per-instance data, while the *instance* namespace lists data and no methods. This contrast is the cleanest demonstration of the scoping rule from Session 12: lookups start at the instance, then fall back to the class — so `p.address()` finds the method in the class while `p.__dict__` finds the data attached to the object.

Note the mangled keys `_Person__city` and `_Person__age` in the output — confirming the private attributes are stored on the instance, just under their renclared names. The address printed is `Raj, Kolkata, West Bengal`, showing the public `name`, the mangled private `city` (set via the setter), and the public `state`.

In [ ]:
# Write your code here
p = Person("Raj", "West Bengal")
p.set_city("Kolkata")
p.set_age(40)
print("Address: ", p.address())

for i in p.__dict__:
    print(i)

Address:  Raj, Kolkata, West Bengal
name
state
_Person__city
_Person__age


### `Q3:` Write a recursive program to to calculate `gcd` and print no. of function calls taken to find the solution.
```
gcd(5,10) -> result in 5 as gcd and function call 3
```

### Q3 — recursion depth surfaced with a global counter

The question mixes **recursion** with **counting function calls**. The solution defines a module-level `counter = 0`, then a `gcd` function that bumps a `global counter` on every entry. Each recursive invocation `gcd(b, a % b)` applies **Euclid's algorithm**: the GCD of two numbers is unchanged when the larger is replaced by its remainder on division by the smaller. When the remainder reaches 0, `a` is the GCD.

```python
counter = 0
def gcd(a, b):
    global counter
    counter += 1
    if b == 0:
        return a
    return gcd(b, a % b)
```

`gcd(10, 5)` produces `5` with `2` calls: the first call sees `b = 5`, recurses into `gcd(5, 0)`, and the second call returns `5`. The `global counter` declaration is mandatory — without it, `counter += 1` would create a *local* variable on first use and raise `UnboundLocalError`. This is a nice miniature of scope rules in action: the recursive frame stack is exactly the place where local names shadow outer ones, and `global` is how you deliberately reach out to the module namespace.

In [ ]:
#Write your code here
# O(log(min(a,b)))

counter = 0
def gcd(a,b):
    global counter
    counter += 1
    if b == 0:
        return a
    else:
        return gcd(b, a%b)
print(gcd(10, 5), counter)

5 2


## Iterator And Generator

### `Q4:` Create MyEnumerate class, 
Create your own `MyEnumerate` class such that someone can use it instead of enumerate. It will need to return a `tuple` with each iteration, with the first element in the tuple being the `index` (starting with 0) and the second element being the `current element` from the underlying data structure. Trying to use `MyEnumerate` with a noniterable argument will result in an error.

```
for index, letter in MyEnumerate('abc'):
    print(f'{index} : {letter}')
```

Output:
```
0 : a
1 : b
2 : c
```

### Q4 — a hand-built `MyEnumerate`, an iterator class

`enumerate` pairs each element of an iterable with its index starting at 0. The custom `MyEnumerate` reproduces it by implementing the full **iterator protocol** as a class:

- `__init__` stores the data and an `index` server counter.
- `__iter__` returns `self`, advertising that this object is its own iterator.
- `__next__` checks whether `index` has reached the end; if so it **raises `StopIteration`** — the universal "no more items" signal — otherwise it returns the tuple `(index, data[index])` and advances the counter.

Because a `for` loop silently consumes `StopIteration`, the loop

```python
for index, letter in MyEnumerate('abc'):
    print(f'{index} : {letter}')
```

produces `0 : a`, `1 : b`, `2 : c` exactly like `enumerate`. This is the canonical shape every Python iterator follows, and it is worth memorising:

```python
def __iter__(self):
    return self
def __next__(self):
    if self.index >= len(self.data):
        raise StopIteration
    value = (self.index, self.data[self.index])
    self.index += 1
    return value
```

`list(MyEnumerate('ab'))` would also work because `list()` consumes an iterable the same way a `for` loop does.

In [ ]:
#Write your code here
   
class MyEnumerate:
    def __init__(self, data):
        self.data = data
        self.index = 0
    def __iter__(self):
        return self
    def __next__(self):
        if self.index >= len(self.data):
            raise StopIteration
        value = (self.index, self.data[self.index])
        self.index += 1
        return value
for index, letter in MyEnumerate('abc'):
    print(f'{index} : {letter}')

0 : a
1 : b
2 : c


### `Q5:` Iterate in circle
Define a class, `Circle`, that takes two arguments when defined: a sequence and a number. The idea is that the object will then return elements the defined number of times. If the number is greater than the number of elements, then the sequence  repeats as necessary. You can define an another class used as a helper (like I call `CircleIterator`).

```
c = Circle('abc', 5)
d = Circle('abc', 7)
print(list(c))
print(list(d))
```

Output
```
[a, b, c, a, b]
[a, b, c, a, b, c, a]
```

### Q5 — `Circle`: cycling a sequence with a bound

`Circle('abc', 5)` must yield `a, b, c, a, b`, and with bound 7 it must yield the sequence plus `a` again. The trick is to reuse a *single* counter and map it onto the sequence with the modulo operator: at iteration `i` the item is `data[i % len(data)]`. The iterator class stores three pieces of state — the data, an index counter starting at 0, and the total number of iterations allowed.

```python
def __next__(self):
    if self.index >= self.max_iters:
        raise StopIteration
    value = self.data[self.index % len(self.data)]
    self.index += 1
    return value
```

`__iter__` returns `self`, so the object is directly consumable by `list()`. The question allows a helper class (`CircleIterator`) but the expected solution shows how compact the "self-iterator" pattern can be. The modulo is the heart: `0 % 3 -> a`, `1 % 3 -> b`, `2 % 3 -> c`, `3 % 3 -> a`, `4 % 3 -> b`, giving exactly five elements for the first case. Note the printed output uses quote marks (`['a', 'b', ...]`) because `list()` stringifies every yielded element with `repr`.

In [ ]:
#Write your code here
class Circle:
    def __init__(self, data, max_iters):
        self.data = data
        self.index = 0
        self.max_iters = max_iters
    
    def __iter__(self):
        return self
    def __next__(self):
        if self.index >= self.max_iters:
            raise StopIteration
        value = self.data[self.index % len(self.data)]
        self.index += 1
        return value
c = Circle('abc', 5)
d = Circle('abc', 7)
print(list(c))
print(list(d))


['a', 'b', 'c', 'a', 'b']
['a', 'b', 'c', 'a', 'b', 'c', 'a']


### `Q6:` Generator time elapsed
Write a generator function whose argument must be iterable. With each iteration, the generator will return a two-element tuple. The first element in the tuple will be an integer indicating how many seconds have passed since the previous iteration. The tuple’s second element will be the next item from the passed argument.

Note that the timing should be relative to the previous iteration, not when the
generator was first created or invoked. Thus the timing number in the first iteration
will be 0

```
for t in elapsed_since('abcd'):
    print(t)
    time.sleep(2)
```

Output:
```
(0.0, 'a')
(2.005651817999933, 'b')
(2.0023095009998997, 'c')
(2.001949742000079, 'd')
```
Note: Your output may differ because of diffrent system has different processing configuration.

### Q6 — `elapsed_since`: a generator measuring time between iterations

The challenge here is relative timing: each yielded pair is `(seconds since the previous yield, next item)`, and the *first* pair must report `0.0`. A generator function fits perfectly because it preserves its local variables between calls. The solution stores `last_time = time.perf_counter()` before the loop, then for each item measures a fresh timestamp, computes `delta = current_time - last_time`, stores the current time as the new baseline, and yields `(delta, item)`.

```python
def elapsed_since(data):
    last_time = time.perf_counter()
    for item in data:
        current_time = time.perf_counter()
        delta = current_time - last_time
        last_time = current_time
        yield (delta, item)
```

`time.perf_counter()` is the highest-resolution clock and is meant precisely for such measurements. In the driver, `time.sleep(2)` between prints makes each subsequent delta come to about 2 seconds — the tiny leading value is the microsecond overhead of the first measurement, so the first delta is not exactly 0 but a very small number. Your outputs will differ slightly because timings depend on the machine; the structure — not the decimal — is what is being tested. This is a great example of why generators are "lazy state machines": the state (`last_time`) lives in the generator frame between `yield` resumes.

In [ ]:
#Write yor code
import time
def elapsed_since(data):
    last_time = time.perf_counter()
    for item in data:
        current_time = time.perf_counter()
        delta = current_time - last_time
        last_time = current_time
        yield (delta, item)

for t in elapsed_since('abcd'):
    print(t)
    time.sleep(2)

(1.3999999737279722e-06, 'a')
(2.005139500000041, 'b')
(2.0140556999999717, 'c')
(2.0137271000000965, 'd')


## Decorators

### `Q7:` Write a Python program to make a chain of function decorators (bold, italic, underline etc.) on a given function which prints "hello world"

```
def hello():
    return "hello world"
```

```
bold - wrap string with <b> tag. <b>Str</b>
italic - wrap string with <i> tag. <i>Str</i>
underline- wrap string with <u> tag. <u>Str</u>
```

### Q7 — a chain of decorators: bold, italic, underline

Decorators wrap one function inside another. Each of `make_bold`, `make_italic`, and `make_underline` follows the identical pattern: it receives the original function, defines an inner closure that calls it and wraps the result in the matching HTML tag, and returns the closure instead of the original.

```python
def make_bold(func):
    def wrapped():
        return "<b>" + func() + "</b>"
    return wrapped
```

Applying three decorators to one function,

```python
@make_bold
@make_italic
@make_underline
def hello():
    return "hello world"
```

is equivalent to `hello = make_bold(make_italic(make_underline(hello)))` — the bottommost decorator (`make_underline`) is applied first, so it wraps the innermost text, and each decorator above it adds another layer outside. That is why the output is `<b><i><u>hello world</u></i></b>`: `underline` is innermost, then `italic`, then `bold`. This "bottom-up, outside-in" ordering is the classic source of confusion with decorator chains, and it is exactly what Q7 exists to test.

In [ ]:
#Write your code here
def make_bold(func):
    def wrapped():
        return "<b>"+ func() + "</b>"
    return wrapped
def make_italic(func):
    def wrapped():
        return "<i>"+ func() + "</i>"
    return wrapped
def make_underline(func):
    def wrapped():
        return "<u>"+ func() + "</u>"
    return wrapped

@make_bold
@make_italic
@make_underline
def hello():
    return "hello world"

print(hello())

<b><i><u>hello world</u></i></b>


### `Q8:` Write a decorator called `printer` which causes any decorated function to print their return values. If the return value of a given function is `None`, printer should do nothing.



### Q8 — the `printer` decorator, with `functools.wraps`

This decorator prints the return value of the wrapped function *only when it is not `None`*. The inner function accept arbitrary arguments with `*args, **kwargs`, calls the original, and checks the result:

```python
def printer(func):
    @wraps(func)
    def inner(*args, **kwargs):
        return_value = func(*args, **kwargs)
        if return_value is not None:
            print(return_value)
        return return_value
    return inner
```

Because the decorated `hello("hello")` returns the string, the decorator prints `hello`; the interpreter line `'hello'` you see in the output is the REPL echoing the returned value. The `@wraps(func)` decorator deserves special attention: without it the metadata of the wrapped function — its `__name__`, `__doc__`, and `__module__` — would be lost and replaced by the closure's own (`inner`). `functools.wraps` copies that metadata across, which is why the very next cell's `help(hello)` still shows the original signature `hello(string)`. This is a professional habit: always decorate your internal closures with `@wraps` unless you have a reason not to.

In [ ]:
#Write your code here
from functools import wraps
def printer(func):
    @wraps(func)
    def inner(*args, **kwargs):
        return_value = func(*args, **kwargs)
        if return_value is not None:
            print(return_value)
        return return_value
    return inner

@printer
def hello(string):
    return string

hello("hello")

hello


'hello'

In [ ]:
help(hello)

Help on function hello in module __main__:

hello(string)



### `Q9:` Make a decorator which calls a given function twice. You can assume the functions don't return anything important, but they may take arguments.
```
#Lets say given function 
def hello(string):
    print(string)

#on calling after specified decorator is inplaced
hello('hello')
```

Output:
```
hello
hello
```

### Q9 — the `double` decorator: calling a function twice

Q9 asks for a decorator that runs the decorated function twice. The solution's inner function calls `func(*args, **kwargs)` twice in a row and implicitly returns `None`:

```python
def double(func):
    @wraps(func)
    def inner(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return inner
```

When `hello("hello")` — a function whose only job is to `print` its argument — is decorated, each call prints the string once, so two calls give:

```
hello
hello
```

Two details are worth noting. First, `*args, **kwargs` makes the wrapper generic, so it works for functions with any number of positional or keyword arguments. Second, the wrapped function has no `return` statement, so `inner` returns `None` — fine here, because the question explicitly says the functions "don't return anything important". This decorator is useful for retries, logging enabled calls, or any behaviour that should happen on both halves of an operation.

In [ ]:
#Write your cod here
def double(func):
    @wraps(func)
    def inner(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return inner
@double
def hello(string):
    print(string)
    
hello("hello")

hello
hello


### `Q10:` Write a decorator which doubles the return value of any function. And test that decoratos is working correctly or not using `asert`.

```
add(2,3) -> result in 10. Without decorator it should be 5.
```

### Q10 — doubling the return value, tested with `assert`

The final decorator multiplies the wrapped function's *return value* by two and returns it:

```python
def double(func):
    @wraps(func)
    def inner(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return inner
```

`add_withDeco(2, 3)` therefore computes `(2 + 3) * 2 = 10`, while the untouched `add(2, 3)` returns `5`. The proof uses `assert`:

```python
assert add(a, b) * 2 == add_withDeco(a, b), "Values are not matching"
```

If the decorator were broken, the comparison would be false and the assertion would raise `AssertionError` with the message "Values are not matching"; because it passes, the next line prints `Values are matching`. This is a good template for *verifying* decorators: compare the decorated output against the expected value expressed with the plain function, rather than trusting the decorated test by eye.

In [ ]:
# Write your code here
def double(func):
    @wraps(func)
    def inner(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return inner
@double
def add_withDeco(a,b):
    return a+b

def add(a,b):
    return a+b

a=2
b=3
assert add(a,b)*2 == add_withDeco(a,b), "Values are not matching"
print("Values are matching")

Values are matching


## Summary

This solutions sheet walked through every Task 12 program. On **namespaces and scope**, you saw that `Class.__dict__` lists methods and class machinery while `instance.__dict__` lists the object's own data — private attributes appearing under their mangled names (`_Person__city`) — and that a `global` declaration is what lets recursive code count its own calls. On **iterators and generators**, `MyEnumerate` and `Circle` re-implemented the protocol by hand (`__iter__` returns `self`, `__next__` raises `StopIteration`), while `elapsed_since` used a generator's lazy, stateful frame to measure time between consecutive yields. On **decorators**, you stacked three wrappers bottom-up (`<u>` innermost, `bold` outermost), printed return values conditionally, called functions twice, and doubled a return value — each closure made reusable with `*args, **kwargs` and `@functools.wraps`, and the double decorator verified with an `assert`. After running these cells end to end, you should be able to explain *when* each tool is the right one: an iterator class when you need full control of state, a generator when you want lazy one-shot traversal, and a decorator whenever you want to enrich a function without editing its body.